## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [1]:
# basic hello world code


In [2]:
# The imports
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    Runner,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
    set_tracing_export_api_key,
    trace,
)

In [3]:
# The usual starting point
load_dotenv(override=True)

True

In [4]:
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
client = AsyncOpenAI(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
)

set_default_openai_api("chat_completions")
set_default_openai_client(client, use_for_tracing=False)

# OpenRouter runs inference; export traces to platform.openai.com with a real OpenAI key
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    set_tracing_export_api_key(openai_api_key)
else:
    set_tracing_disabled(True)

In [5]:
# Make an agent with name, instructions, model
model = OpenAIChatCompletionsModel(model="gpt-4.1-nano", openai_client=client)

In [6]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model=model)

In [7]:
# Run the joke with Runner.run(agent, prompt) then print final_output
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the autonomous AI agent get promoted?

Because it always took the initiative without needing direction!


In [10]:
# Explore the result object to see what trace info is available
print("Result object type:", type(result))
print("Result attributes:", dir(result))
print("\nResult details:")
print(f"- final_output: {result.final_output}")
if hasattr(result, 'messages'):
    print(f"- messages: {result.messages}")
if hasattr(result, 'trace_id'):
    print(f"- trace_id: {result.trace_id}")

Result object type: <class 'agents.result.RunResult'>
Result attributes: ['__abstractmethods__', '__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_last_agent', 'context_wrapper', 'final_output', 'final_output_as', 'input', 'input_guardrail_results', 'last_agent', 'last_response_id', 'new_items', 'output_guardrail_results', 'raw_responses', 'to_input_list', 'tool_input_guardrail_results', 'tool_output_guardrail_results']

Result details:
- final_output: Why did the autonomous AI agent get promoted?

Because it always took the initiative without needing direction!


In [11]:
import json

# Display trace information from the result
print("=" * 60)
print("TRACE INFORMATION")
print("=" * 60)

print(f"\n📥 Input: {result.input}")
print(f"\n📤 Final Output:\n{result.final_output}")

print(f"\n🔄 Raw Responses Count: {len(result.raw_responses)}")
print("\nRaw Responses:")
for i, response in enumerate(result.raw_responses, 1):
    print(f"\n  Response {i}:")
    print(f"    Type: {type(response)}")
    print(f"    Content: {response}")

if hasattr(result, 'last_response_id'):
    print(f"\n📋 Last Response ID: {result.last_response_id}")

if hasattr(result, 'new_items') and result.new_items:
    print(f"\n📝 New Items: {len(result.new_items)}")
    for item in result.new_items:
        print(f"    - {item}")

TRACE INFORMATION

📥 Input: Tell a joke about Autonomous AI Agents

📤 Final Output:
Why did the autonomous AI agent get promoted?

Because it always took the initiative without needing direction!

🔄 Raw Responses Count: 1

Raw Responses:

  Response 1:
    Type: <class 'agents.items.ModelResponse'>
    Content: ModelResponse(output=[ResponseOutputMessage(id='__fake_id__', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent get promoted?\n\nBecause it always took the initiative without needing direction!', type='output_text', logprobs=None)], role='assistant', status='completed', type='message')], usage=Usage(requests=1, input_tokens=23, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=20, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=43), response_id=None)

📋 Last Response ID: None

📝 New Items: 1
    - MessageOutputItem(agent=Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp

## Now go and look at the trace

https://platform.openai.com/traces

When using OpenRouter for inference, set `OPENAI_API_KEY` in `.env` so traces can be exported to the OpenAI platform. Without it, tracing is disabled to avoid non-fatal 401 errors from sending an OpenRouter key to OpenAI's trace endpoint.